[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/14_evaluacion_online/14_evaluacion_online.ipynb)

# Módulo 14 · Evaluación online: ¿las respuestas son buenas?, medido en producción

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 4 h 30 min (lección ≈ 2 h 45 min + proyecto ≈ 1 h 45 min) |
| **Servicios de Compose** | Opcional: `--profile core --profile metrics` (sección 8 y paso final). Sin Docker todo funciona en **modo degradado** (telemetría en memoria, mismo esquema). Funciona en Colab |
| **RAM aproximada** | < 500 MB sin stack · ≈ 1,2 GB con `core` + `metrics` |
| **Prerrequisitos** | Módulos 04 (GenAI semconv), 08 (Loki), 09 (métricas), 10 (Grafana) y 13 (coste por ticket) |
| **Hilo conductor** | SupportOps responde rápido y barato… pero ¿responde **bien**? Ningún dashboard de latencia ni de errores lo sabe |

> ⚠️ **Honestidad sobre el laboratorio.** El agente y el «juez» de este módulo son **simulados y deterministas** (`mock`): el juez es un conjunto de reglas con errores
> realistas (falsos negativos, falsos positivos, sesgo de verbosidad y de posición) para que **puedas medir sus defectos contra la verdad de terreno**, cosa que con un
> LLM real hay que hacer con etiquetas humanas. Las cifras cambian con un juez real; el **método** no.

## Objetivos de aprendizaje

Al terminar esta lección podrás:

1. **Distinguir** evaluación *offline* y *online*, y elegir el tipo de evaluador (reglas, juez LLM, humano, feedback) según coste, latencia y fiabilidad.
2. **Implementar a mano** un evaluador de *groundedness* (afirmaciones respaldadas por el contexto) y medir su sensibilidad y especificidad contra la verdad.
3. **Emitir** una evaluación como telemetría OpenTelemetry: evento `gen_ai.evaluation.result` (spec GenAI, estado *Development*) asociado al span evaluado, span del juez enlazado a la traza original y métricas.
4. **Diseñar** un muestreo estratificado para evaluar y **estimar** la calidad poblacional sin sesgo (post-estratificación) con su intervalo de confianza.
5. **Detectar y cuantificar** los sesgos del juez (verbosidad, posición) y **corregir** una tasa observada con la sensibilidad/especificidad del juez (Rogan–Gladen).
6. **Interpretar** el feedback de usuario como señal sesgada por la tasa de respuesta, y combinarlo con el juez.
7. **Calcular** el coste de evaluar y elegir una tasa de muestreo que cumpla un presupuesto y una precisión.
8. **Construir** un panel de Grafana de calidad online con sus consultas (PromQL, LogQL, TraceQL).

## 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 04) ¿En qué estado está la spec GenAI y por qué importa para los nombres que usaremos hoy?</summary>

Está en estado *Development* (inestable): los nombres pueden cambiar. Por eso en SupportOps los atributos se escriben como cadenas verificadas contra una instantánea de la spec y no se importan de librerías que van por detrás.

</details>

<details>
<summary>2 · (Módulo 13) ¿Dónde se suman los tokens y el coste de un ticket sin duplicar?</summary>

Solo en los spans hoja `chat`. Hoy esto tiene una consecuencia: el **juez** también hace llamadas `chat`; por eso vive en **otro servicio** (`supportops-evaluator`) y su coste no se mezcla con el del agente.

</details>

<details>
<summary>3 · (Módulo 09) ¿Por qué `ticket_id` no puede ser una etiqueta de una métrica?</summary>

Cada valor distinto crea una serie temporal nueva: cardinalidad ilimitada. Los identificadores van en spans/logs; las métricas llevan etiquetas acotadas (estrato, criterio, etiqueta pass/fail, versión).

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import dataclasses, json, math, re, time
from supportops.instrument import CompositeInstrumentor
from supportops.telemetry import OTelInstrumentor, setup_telemetry
from supportops.quality_eval import (EVALUATORS, EVENT_NAME, CaseCollector, EvalCase, EvalScore, LLMJudge, MockJudge, OnlineEvaluator, RateSampler, RuleJudge,
                                     StratifiedSampler, build_eval_event, confusion, extract_claims, find_pii, post_stratified_rate, render_judge_prompt,
                                     required_sample_size, rogan_gladen, scrub_pii, simulate_feedback, unsupported_claims, wilson_interval)
STACK = check_stack("core+metrics")
LIVE = not STACK.degraded                      # True: Collector + Prometheus responden y la sección 8 envía telemetría de verdad

In [ ]:
N = 600 if FAST_DEV_RUN else 3000
FAULTS = {**DEFAULT_FAULT_RATES, "hallucination": 0.06}            # alucinaciones al 6 % (el doble que la base) para tener qué detectar
TICKETS = generate_tickets(N, seed=SEED, fault_rates=FAULTS)
col = CaseCollector()                                                 # un EvalCase por ticket: lo que verá el juez + ground truth APARTE
RESULTS = run_tickets(SupportAgent(instrumentor=col), TICKETS)
CASES = col.cases
RULE, JUDGE = RuleJudge(), MockJudge(SEED)
print(f"{len(CASES)} casos · alucinaciones reales: {sum(not c.truth['grounded'] for c in CASES)} · fugas de PII: {sum(c.truth['pii_leaked'] for c in CASES)} · "
      f"tickets sin resolver: {sum(c.status in ('error', 'max_steps', 'degraded') for c in CASES)}")

---
# 1 · Intuición: el dashboard verde que miente

Imagina el panel de salud de SupportOps (módulo 10): latencia p95 estable, 0,8 % de errores, coste por ticket normal… y el agente lleva una hora **inventándose el número de
seguimiento de los pedidos**. Ninguna señal clásica lo ve: la petición *terminó bien* (HTTP 200, `status=ok`) y rápido. **La calidad es una propiedad del contenido**, no de la ejecución.

| Si vienes de… | Esto es como… | Lo que cambia |
|---|---|---|
| **ML / ciencia de datos** | Evaluación *offline* con un conjunto de test vs monitorizar el modelo en producción | No hay etiqueta de verdad por petición: hay que **estimarla** (reglas, juez, humanos, feedback) y **muestrear** porque evaluar cuesta |
| **Backend** | Un *health check* que devuelve 200 pero la base de datos devuelve datos corruptos | El «200» no prueba que el resultado sea correcto: necesitas una **comprobación semántica** (un *canary* funcional) |
| **SRE** | SLI de calidad de servicio: no solo «¿responde?» sino «¿responde bien?» | La calidad se convierte en un SLI con su SLO y su presupuesto (módulo 16) |

El patrón de la industria: **la evaluación online es otro consumidor de la telemetría**. La traza ya contiene la pregunta, el contexto recuperado y la respuesta; un evaluador asíncrono
(fuera del camino crítico de la petición) lee una **muestra**, la puntúa y publica el resultado **asociado al span evaluado**, de modo que desde una puntuación mala saltas a la traza que la causó.

In [ ]:
fig, ax = viz.canvas(12.5, 4.9, (0, 125), (0, 48))
P = viz.PALETTE
b_u = viz.box(ax, 1, 20, 13, 9, "Cliente\n(ticket)", P["neutral"])
b_a = viz.box(ax, 20, 20, 17, 9, "Agente SupportOps\n(invoke_agent)", P["traces"], bold=True)
b_r = viz.box(ax, 43, 20, 15, 9, "Respuesta\n(span chat)", P["traces"])
b_t = viz.box(ax, 43, 36, 15, 8, "Trazas · logs\n(Tempo / Loki)", P["logs"])
b_s = viz.box(ax, 64, 36, 17, 8, "Muestreo\npor estratos", P["warn"])
b_j = viz.box(ax, 64, 20, 17, 9, "Juez\n(reglas / LLM)", P["profiles"], bold=True)
b_e = viz.box(ax, 88, 33, 17, 9, "Evento\ngen_ai.evaluation.result", P["logs"], fontsize=8)
b_m = viz.box(ax, 88, 20, 17, 9, "Métricas\nsupportops.eval.*", P["metrics"])
b_v = viz.box(ax, 88, 7, 17, 9, "Span\nsupportops.evaluate", P["traces"], fontsize=8)
b_g = viz.box(ax, 110, 20, 14, 9, "Grafana\n+ alertas / SLO", P["dark"], fc="#e5e7eb")
b_f = viz.box(ax, 20, 4, 17, 8, "Feedback del usuario\n(pulgar ↑/↓, sesgado)", P["warn"], fontsize=8)
for a, b in ((b_u, b_a), (b_a, b_r), (b_s, b_j)):
    viz.arrow(ax, viz.right(a), viz.left(b))
viz.arrow(ax, viz.right(b_r), viz.left(b_j), "", P["neutral"], dashed=True)
viz.arrow(ax, viz.top(b_r), viz.bottom(b_t), "", P["logs"])
viz.arrow(ax, viz.right(b_t), viz.left(b_s))
viz.arrow(ax, viz.right(b_j), viz.left(b_m)); viz.arrow(ax, viz.top(b_j), (88 - 0.3, 36), "", P["logs"], rad=-0.2)
viz.arrow(ax, (b_j[0] + b_j[2], 20), (88 - 0.3, 11.5), "", P["traces"], rad=0.25)
viz.arrow(ax, viz.right(b_m), viz.left(b_g)); viz.arrow(ax, viz.right(b_e), (118, 29.5), "", P["logs"], rad=-0.2)
viz.arrow(ax, viz.right(b_f), (60, 8), "", P["warn"], dashed=True)
ax.text(62, 4.2, "→ feedback_total", fontsize=8, color=P["warn"])
ax.text(1, 45, "La evaluación online NO está en el camino crítico: ocurre DESPUÉS, sobre una muestra", fontsize=10, fontweight="bold")
plt.show()

> 👀 **Qué debes observar**
>
> - El juez **no** está entre el cliente y la respuesta: no añade latencia al usuario. Lee la traza ya terminada (flecha discontinua) y publica **tres señales**: un evento por criterio (logs), una métrica y un span propio.
> - El **muestreo** está antes del juez porque evaluar cuesta dinero (sección 9). La pregunta de diseño central es *qué evaluar*, no *cómo puntuar*.
> - El feedback del usuario es una señal **distinta y barata** (sección 8), pero tiene un sesgo de respuesta: lo miraremos con números.

## 1.1 La demostración: calidad invisible para las señales clásicas

Comparamos, ticket a ticket, lo que ven los paneles clásicos (estado, latencia, coste) con la verdad que conocemos por construcción (el inyector sabe qué respuestas son inventadas).

In [ ]:
df = pd.DataFrame(to_rows(TICKETS, RESULTS))
df["bad_truth"] = [c.truth_bad for c in CASES]
df["halluc"] = [not c.truth["grounded"] for c in CASES]
h = df[df.halluc]; ok = df[(~df.halluc) & (df.status == "ok")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col_, ttl, unit in ((axes[0], "latency_s", "Latencia por ticket (s)", "s"), (axes[1], "cost_usd", "Coste por ticket (USD mock)", "$"), (axes[2], "output_tokens", "Tokens de salida", "tok")):
    ax.boxplot([ok[col_], h[col_]], tick_labels=["correctas (ok)", "alucinadas"], showfliers=False, patch_artist=True,
               boxprops={"facecolor": viz.PALETTE["light"]}, medianprops={"color": viz.PALETTE["error"]})
    ax.set_title(ttl)
plt.tight_layout(); plt.show()
print("estado de las alucinadas:", h.status.value_counts().to_dict(), "· tickets 'error' entre las alucinadas:", int((h.status == "error").sum()))
print("medianas (correctas vs alucinadas): latencia %.2f vs %.2f s · coste %.6f vs %.6f USD · tokens de salida %d vs %d" % (ok.latency_s.median(), h.latency_s.median(), ok.cost_usd.median(), h.cost_usd.median(), ok.output_tokens.median(), h.output_tokens.median()))

> 👀 **Qué debes observar**
>
> - Las respuestas alucinadas son **más rápidas y baratas** que las correctas (no llaman a ninguna herramienta, contestan al primer intento) y su estado es `ok`. Si optimizas latencia y coste, **premias** al agente que se inventa los datos.
> - Es el patrón de las *regresiones silenciosas* del módulo 15: mejoran los números que miramos y empeoran los que no medimos.
> - Conclusión: hay que **medir el contenido**. El resto del módulo es cómo hacerlo barato, sin sesgo y observable.

---
# 2 · Teoría: qué evaluar y con qué

## 2.1 Criterios (qué significa «bien»)

| Criterio | Pregunta | Necesita… | En SupportOps |
|---|---|---|---|
| **Groundedness / fidelidad** | ¿Cada dato de la respuesta está respaldado por el contexto (resultados de herramientas)? | la respuesta + el contexto | Detecta alucinaciones (importes, referencias, plazos inventados) |
| **Relevancia** | ¿Responde a lo que se preguntó? | pregunta + respuesta | Pide un dato que ya dio el cliente → falla |
| **Resolución** | ¿Resuelve o encamina bien el caso? | respuesta + estado | Mensajes de *fallback* no resuelven |
| **Seguridad / PII** | ¿Revela datos personales ajenos? | respuesta | Regex deterministas: **no** se delega en un LLM |

«Sin referencia» (*reference-free*): en producción no hay respuesta de oro; se evalúa la respuesta frente a **su contexto y su pregunta**. Las etiquetas de oro viven en el dataset *offline* (módulo 15).

## 2.2 Tipos de evaluador

| Tipo | Coste | Latencia | Fiabilidad | Úsalo para… |
|---|---|---|---|---|
| **Reglas / código** (regex, esquemas, comprobaciones numéricas) | ≈ 0 | µs–ms | Exacta en lo que cubre | PII, formato, ids inventados, *todo lo verificable por código* |
| **Juez LLM** | alto (1 llamada por criterio o por caso) | s | Buena pero **sesgada** (verbosidad, posición, autopreferencia) | Criterios semánticos que el código no ve (tono, relevancia, razonamiento) |
| **Humano** | muy alto | horas–días | La referencia, con desacuerdo entre anotadores | Calibrar al juez y construir el dataset de oro |
| **Feedback de usuario** | ≈ 0 | inmediata | Escasa y **sesgada** por quién responde | Señal de alarma y fuente de casos difíciles |

Regla práctica: **código primero, LLM para lo que el código no puede, humanos para calibrar** (Husain, *Your AI Product Needs Evals*; Yan, *Evaluating the Effectiveness of LLM-Evaluators*).

## 2.3 Qué dice la spec de OpenTelemetry (estado real, oct-2026)

La spec **GenAI semantic conventions** (estado *Development*, repo `open-telemetry/semantic-conventions-genai`) define un **evento** `gen_ai.evaluation.result`:
«captura el resultado de evaluar la salida de GenAI en calidad, exactitud u otras características; **DEBERÍA** tener como padre el span de la operación evaluada o, si no hay id de span, llevar `gen_ai.response.id`». Atributos:
`gen_ai.evaluation.name` (*Required*), `gen_ai.evaluation.score.value` (double) y `gen_ai.evaluation.score.label` (*Conditionally Required*), `gen_ai.evaluation.explanation` y `gen_ai.response.id` (*Recommended*), `error.type` (si la evaluación falló).

**Lo que la spec NO define** (por tanto es convención propia, y así lo marcamos con el prefijo `supportops.`): un span para el evaluador, métricas de calidad, el estrato de muestreo, el feedback del usuario.

In [ ]:
snap = json.load(open(ROOT / "04_genai_semconv" / "genai_semconv_snapshot.json"))
print("Instantánea de la spec:", snap["source"]["repo"].split("/")[-1], "· commit", snap["source"]["commit"][:8], "·", snap["source"]["date"], "·", snap["source"]["status"])
rows = [{"atributo": k, "tipo": snap["attributes"][k]["type"], "estabilidad": snap["attributes"][k]["stability"], "descripción": snap["attributes"][k]["brief"][:80]}
        for k in ("gen_ai.evaluation.name", "gen_ai.evaluation.score.value", "gen_ai.evaluation.score.label", "gen_ai.evaluation.explanation", "gen_ai.prompt.name", "gen_ai.prompt.version", "gen_ai.agent.version")]
display(pd.DataFrame(rows).set_index("atributo"))

> **⚠️ Cuidado · Cuidado**
>
> Todo lo de `gen_ai.evaluation.*` es **Development**: puede cambiar de nombre o forma (hasta hace poco la evaluación se modelaba de otra manera en algunas herramientas). Aísla los nombres
> en **una** función (`build_eval_event`) y valida contra la instantánea de la spec en CI, como hicimos con el linter del módulo 04.

---
# 3 · A mano: un evaluador de groundedness con reglas

Antes de usar un LLM como juez, veamos hasta dónde llega **el código**. Una alucinación típica de SupportOps es un dato concreto (referencia de reembolso, importe, tracking, plazo) que **no aparece** en lo que devolvieron las herramientas. Eso se puede comprobar con expresiones regulares:

1. **Extraer afirmaciones** verificables de la respuesta: identificadores (`ORD-…`, `TRK…`, `RF-…`), importes en € y «número + unidad» (años, días, unidades).
2. **Comprobar** cada una contra el contexto (JSON de las herramientas) y contra lo que dijo el propio cliente.
3. Si alguna no está respaldada → `groundedness = 0`.

In [ ]:
ej = [c for c in CASES if not c.truth["grounded"]][:2] + [c for c in CASES if c.truth["grounded"] and c.n_tool_calls >= 1][:2]
for c in ej:
    print(f"[{'ALUCINADA' if not c.truth['grounded'] else 'correcta  '}] {c.answer[:110]}")
    print("    afirmaciones:", extract_claims(c.answer), "· sin respaldo:", unsupported_claims(c.answer, c.context, c.question))

In [ ]:
def groundedness_by_hand(case: EvalCase) -> float:
    """1.0 si toda afirmación verificable de la respuesta está respaldada por el contexto o la pregunta; 0.0 si no."""
    return 0.0 if unsupported_claims(case.answer, case.context, case.question) else 1.0

pred_bad = [groundedness_by_hand(c) == 0.0 for c in CASES]
truth_bad = [not c.truth["grounded"] for c in CASES]
cm = confusion(pred_bad, truth_bad)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in cm.items()})

> 👀 **Qué debes observar**
>
> - Con **cero** llamadas a un LLM el evaluador de reglas encuentra **todas** las alucinaciones (sensibilidad 1,0) y no acusa a ninguna respuesta correcta (especificidad 1,0). En este laboratorio las alucinaciones son datos inventados *comprobables*.
> - **Cuidado con extrapolar**: en la vida real hay alucinaciones que el código no ve (un razonamiento falso sin cifras, una paráfrasis que cambia el significado). Para esas hace falta un juez semántico — y ahí empiezan los errores de medición.
> - Las reglas también se equivocan si olvidas casos: mi primera versión marcaba como inventado el ejemplo de formato `ORD-00000` y el plazo de política «5 días» — los **falsos positivos** de una regla se corrigen añadiendo excepciones, y por eso se mide.

---
# 4 · El juez: anatomía, protocolo y proveedores

Un **LLM-as-judge** es un LLM al que le pasas la rúbrica, la pregunta, el contexto y la respuesta, y te devuelve puntuaciones estructuradas. Lo importante para observabilidad:

- Es una **llamada `chat` más**: tiene tokens, latencia, coste, fallos y *rate limits*. Hay que instrumentarla.
- Su salida es **texto**: puede no ser JSON válido. Un juez que falla en silencio sesga la métrica hacia «todo bien» → se cuenta `supportops.eval.errors`.
- Es **agnóstico de proveedor**: usa la misma capa `supportops.llm` (`JUDGE_PROVIDER=mock|anthropic|openai|ollama` y `JUDGE_MODEL`); aquí corre el `MockJudge` determinista.

In [ ]:
case = next(c for c in CASES if not c.truth["grounded"] and c.n_tool_calls == 0)
prompt = render_judge_prompt(case)
print(prompt)
print("-" * 80, "\ntokens del prompt del juez ≈", len(prompt) // 4, "· tokens del agente en ese ticket =", next(r.input_tokens for r in RESULTS if r.ticket_id == case.ticket_id))
v = JUDGE.evaluate(case)
for name, s in v.scores.items():
    print(f"{name:13s} {s.value:5.2f} {s.label:5s} {s.explanation}")
print(f"juez: {v.judge_model} · {v.input_tokens} tok in / {v.output_tokens} tok out · latencia simulada {v.latency_s:.2f}s · coste ${v.cost_usd:.5f} (ilustrativo)")

**Contrato de implementación** (`LLMJudge`): enviamos el prompt con `temperature=0`, pedimos solo JSON y lo parseamos de forma tolerante; si no se puede parsear devolvemos un veredicto con `error="parse_error"` en lugar de inventar una nota.

In [ ]:
stub = type("StubLLM", (), {"model": "mock-large"})()          # cualquier cliente de supportops.llm vale; aquí solo probamos el parseo
lj = LLMJudge(stub)
good = lj.parse("T-1", 'Claro: {"groundedness": {"score": 0.9, "explanation": "ok"}, "relevance": {"score": 1}, "resolution": {"score": 1}, "pii_safety": {"score": 1}}')
bad = lj.parse("T-2", "Lo siento, no puedo evaluar esto.")
print("válido  ->", good.error, good.passed, {k: v.value for k, v in good.scores.items()})
print("inválido->", bad.error, bad.passed, bad.scores)

> 👀 **Qué debes observar**
>
> - El prompt del juez es **más largo que la conversación del agente** (lleva rúbrica + contexto + respuesta): el juez cuesta más por caso que el agente. Lo cuantificamos en la sección 9.
> - El parseo defensivo es parte de la observabilidad del juez: sin él, un cambio de formato del modelo juez parece «de repente todo va perfecto».
> - **Privacidad**: lo que mandas al juez (a menudo un tercero) incluye pregunta, contexto y respuesta, con su PII. `scrub_pii()` redacta emails, teléfonos y tarjetas antes de enviar; y la detección de PII se hace con regex locales, no con el LLM.

---
# 5 · Los errores del juez: medirlos, no ignorarlos

El `MockJudge` se equivoca a propósito de cuatro maneras (las que documenta la literatura sobre jueces LLM; Zheng et al., 2023): **falsos negativos** (deja pasar una alucinación), **falsos positivos** (acusa a una respuesta correcta),
**sesgo de verbosidad** (puntúa mejor lo largo) y **sesgo de posición** (en comparaciones por pares prefiere la primera respuesta). Con un juez real hay que *medir* estos errores con un conjunto de calibración etiquetado por humanos; aquí lo etiquetamos con la verdad del inyector.

## 5.1 Sensibilidad, especificidad y qué le pasa a tu métrica

In [ ]:
vs_rule = [RULE.evaluate(c) for c in CASES]; vs_mock = [JUDGE.evaluate(c) for c in CASES]
fail = lambda vs: [v.scores["groundedness"].label == "fail" for v in vs]
cm_r, cm_m = confusion(fail(vs_rule), truth_bad), confusion(fail(vs_mock), truth_bad)
obs_rate = sum(fail(vs_mock)) / len(CASES); true_rate = sum(truth_bad) / len(CASES)
fixed = rogan_gladen(obs_rate, cm_m["sensitivity"], cm_m["specificity"])
print(f"verdad: {true_rate:.2%} alucinadas · observado con el juez: {obs_rate:.2%} · corregido (Rogan–Gladen): {fixed:.2%}")
print(f"juez imperfecto -> sensibilidad {cm_m['sensitivity']:.2f} · especificidad {cm_m['specificity']:.3f} · precisión {cm_m['precision']:.2f}")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, cm_, ttl in ((axes[0], cm_r, "Juez de reglas"), (axes[1], cm_m, "Juez imperfecto (mock)")):
    M = np.array([[cm_["tn"], cm_["fp"]], [cm_["fn"], cm_["tp"]]])
    ax.imshow(M, cmap="Blues"); ax.set_xticks([0, 1], ["dice OK", "dice FALLA"]); ax.set_yticks([0, 1], ["era buena", "era alucinada"]); ax.set_title(ttl); ax.grid(False)
    for (i, j), val in np.ndenumerate(M): ax.text(j, i, int(val), ha="center", va="center", color="white" if val > M.max() / 2 else "black", fontsize=13)
axes[2].bar(["verdad", "observado\n(juez)", "corregido\n(Rogan–Gladen)"], [true_rate, obs_rate, fixed], color=[viz.PALETTE["ok"], viz.PALETTE["warn"], viz.PALETTE["traces"]])
axes[2].set_ylabel("tasa de alucinación"); axes[2].set_title("Tasa de alucinación")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - El juez imperfecto **deja pasar ~10 %** de las alucinaciones y **acusa ~1–2 %** de respuestas correctas. Como las alucinaciones son raras (~7 %), esos falsos positivos son una fracción grande de las alarmas: la **precisión** del juez es mucho menor que su especificidad (la paradoja de la prevalencia baja).
> - La tasa observada **no es** la tasa real. Con sensibilidad *Se* y especificidad *Sp* medidas en un conjunto de calibración, la corrección de Rogan–Gladen recupera la prevalencia: \( \hat\pi = \frac{p_{obs} + Sp - 1}{Se + Sp - 1} \).
> - Esto exige una **muestra etiquetada por humanos** (aquí, la verdad del inyector). Con 100–200 casos etiquetados ya puedes estimar *Se* y *Sp* con intervalo de confianza (`wilson_interval`).

## 5.2 Sesgo de verbosidad: ¿premia el juez lo largo?

In [ ]:
PAD = " Gracias por contactar con SupportOps; estamos encantados de ayudarle con todo lo que necesite y quedamos a su entera disposición para cualquier otra consulta."
sample = [c for c in CASES if c.truth["grounded"] and c.status == "ok" and RULE.relevance(c).value <= 0.7][:200]       # respuestas con margen de mejora (sin techo de 1,0)
print(len(sample), "respuestas correctas pero con relevancia mejorable")
d = []
for k in (0, 1, 2, 3):
    vals = [JUDGE.evaluate(dataclasses.replace(c, answer=c.answer + PAD * k)).scores["relevance"].value for c in sample]
    d.append((k, float(np.mean(vals)), float(np.std(vals) / math.sqrt(len(vals)))))
base = d[0][1]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.errorbar([x[0] * len(PAD) for x in d], [x[1] for x in d], yerr=[1.96 * x[2] for x in d], marker="o", color=viz.PALETTE["profiles"], capsize=4)
ax.set_xlabel("caracteres de relleno añadidos (sin información nueva)"); ax.set_ylabel("puntuación media de relevancia"); ax.set_title("El juez premia el relleno")
plt.tight_layout(); plt.show()
print({k: round(m, 3) for k, m, _ in d})

> 👀 **Qué debes observar**
>
> - Con relleno sin información nueva, la puntuación de relevancia **sube** (de ≈ 0,5 a ≈ 0,65 en este laboratorio): el juez confunde longitud con calidad. Es el sesgo de verbosidad descrito por Zheng et al. (2023) para jueces LLM.
> - Consecuencia operativa: si un cambio de prompt hace al agente más hablador, tu «calidad» sube **sin que el cliente esté mejor atendido**. Defensas: rúbricas que penalicen el relleno, normalizar por longitud, y — sobre todo — **sondas de sesgo** como esta, ejecutadas cada vez que cambias de juez.

## 5.3 Sesgo de posición en comparaciones por pares

In [ ]:
pairs = [(c.question, c.answer) for c in CASES if c.truth["grounded"] and c.status == "ok" and c.n_tool_calls][:120]
same = sum(JUDGE.compare(q, a, a + ".", key=str(i)) == "A" for i, (q, a) in enumerate(pairs))                       # dos respuestas prácticamente IGUALES
swap_consistent = sum((JUDGE.compare(q, a, a + ".", key=str(i)) == "A") != (JUDGE.compare(q, a + ".", a, key=str(i)) == "A") for i, (q, a) in enumerate(pairs))
print(f"con respuestas equivalentes elige A en {same / len(pairs):.0%} de los casos · elige la MISMA respuesta en ambos órdenes: {swap_consistent / len(pairs):.0%} (eligiendo al azar entre equivalentes saldría ≈ 50 %)")

> 👀 **Qué debes observar**
>
> - Ante dos respuestas equivalentes un juez sin sesgo diría «empate» o elegiría al azar (≈ 50 % de «A» en cada orden). Este elige «A» en torno al 60 % de las veces **en ambos órdenes**, así que su veredicto casi nunca es consistente al permutar. La mitigación estándar: evaluar **ambos órdenes** y contar solo los veredictos consistentes (empate si discrepan).
> - Para criterios con respuesta objetiva (¿está respaldado el dato?) prefiere **puntuación directa** a comparación por pares: elimina este sesgo de raíz.

---
# 6 · Muestreo para evaluar: qué evaluar y cómo no sesgar la métrica

Evaluar todo es caro (sección 9). Muestrear es inevitable, pero **un muestreo inteligente sesga la métrica** si no lo corriges:

- **Uniforme** (`RateSampler`): sin sesgo, pero desperdicia presupuesto en tickets aburridos y casi no ve los raros (errores, escalados).
- **Estratificado** (`StratifiedSampler`): evalúa casi todo lo raro e importante (errores 100 %, escalados 50 %, premium 25 %, resto 10 %). Mucho más informativo… y la tasa de «bien» de la **muestra** es engañosa porque sobrerrepresenta los tickets difíciles.

📐 **Estimador post-estratificado.** Con estratos \(s\), población \(N_s\), evaluados \(n_s\) y buenos \(k_s\):
$$\hat p = \sum_s \frac{N_s}{N}\,\frac{k_s}{n_s}, \qquad \widehat{\mathrm{Var}}(\hat p)=\sum_s \Big(\frac{N_s}{N}\Big)^2\frac{\hat p_s(1-\hat p_s)}{n_s-1}\Big(1-\frac{n_s}{N_s}\Big)$$
Cada estrato se pondera por su **tamaño en la población** (`supportops.eval.population` cuenta *todas* las peticiones, evaluadas o no), no por cuántos se evaluaron. Es lo que hace la consulta de PromQL del panel.

In [ ]:
def run_sampler(sampler, judge=RULE):
    ev = OnlineEvaluator(judge, sampler)
    for c in CASES: ev.submit(c)
    ev.run_pending()
    return ev
true_full = float(np.mean([RULE.evaluate(c).passed for c in CASES]))          # lo que daría evaluar TODO con el juez de reglas
rows = []
for s in range(40):
    e1 = run_sampler(StratifiedSampler(salt=f"s{s}")).estimate_quality()
    e2 = run_sampler(RateSampler(0.15, salt=f"s{s}")).estimate_quality()
    rows.append({"estratificado ingenuo": e1["naive"], "estratificado post-estrat.": e1["estimate"], "uniforme 15 %": e2["naive"], "n_estrat": e1["n_evaluated"], "n_unif": e2["n_evaluated"]})
R = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([R[c] for c in ("estratificado ingenuo", "estratificado post-estrat.", "uniforme 15 %")], tick_labels=["estratificado\n(media de la muestra)", "estratificado\n(post-estratificado)", "uniforme 15 %"], patch_artist=True,
           boxprops={"facecolor": viz.PALETTE["light"]}, medianprops={"color": viz.PALETTE["error"]})
ax.axhline(true_full, color=viz.PALETTE["ok"], ls="--", label=f"verdad (evaluando todo) = {true_full:.3f}"); ax.set_ylabel("tasa de calidad OK estimada"); ax.legend(); ax.set_title("40 repeticiones del muestreo con semillas distintas")
plt.tight_layout(); plt.show()
print(R.mean().round(3).to_dict(), "· desviación típica:", R.std().round(3).to_dict())

> 👀 **Qué debes observar**
>
> - La media **ingenua** del muestreo estratificado queda muy por debajo de la verdad: sobrerrepresenta errores y escalados. Si alertaras sobre ella verías una «calidad baja» permanente que es un artefacto del muestreo.
> - El estimador **post-estratificado** cae sobre la verdad (caja centrada en la línea verde) con una dispersión similar a la del muestreo uniforme (≈ 3 puntos), y además **ve los estratos raros**: los errores se evalúan todos (100 %), cosa que un 15 % uniforme no garantiza. Cuesta más evaluaciones (≈ 150 frente a ≈ 90 en el laboratorio), por eso la sección 9.2 pone precio.
> - Regla de oro: **cada muestra lleva su probabilidad de inclusión** (aquí, el estrato). Si no la guardas en la telemetría no podrás corregir después.

In [ ]:
n_pop = {s: v for s, v in run_sampler(StratifiedSampler()).population.items()}
print("población por estrato:", n_pop)
est = run_sampler(StratifiedSampler()).estimate_quality()
lo, hi = est["estimate"] - 1.96 * est["se"], est["estimate"] + 1.96 * est["se"]
print(f"calidad estimada {est['estimate']:.3f}  IC95 [{lo:.3f}, {hi:.3f}]  (n evaluados = {est['n_evaluated']} de {est['n_population']})")
print("n necesario para ±3 puntos con p≈0,85:", required_sample_size(0.85, 0.03), "· para ±1 punto:", required_sample_size(0.85, 0.01))

---
# 7 · Hacerlo observable: evaluación como telemetría OpenTelemetry

Diseño (todo verificado en la sección 8 con el stack real):

| Señal | Qué | Por qué así |
|---|---|---|
| **Evento** `gen_ai.evaluation.result` (log OTel con `event_name`) | uno por criterio, con `name`, `score.value`, `score.label`, `explanation`, `gen_ai.response.id` | La spec pide que cuelgue del **span evaluado** (`trace_id`/`span_id` de la llamada `chat` que produjo la respuesta) |
| **Span** `supportops.evaluate` + hijo `chat <juez>` | el trabajo del juez, con tokens y duración | Es **otra traza** (la evaluación es asíncrona, no es hija de la petición) **enlazada** (*span link*) al span evaluado |
| **Métricas** `supportops.eval.*` | `population`, `sampled`, `results{criterio, etiqueta, estrato, versión}`, `score`, `cost`, `duration` | SLIs y paneles baratos; etiquetas acotadas (nunca `ticket_id`) |
| **Servicio propio** `supportops-evaluator` | evaluador y agente separados | El coste y los tokens del juez no se cuelan en el coste del agente (regla de las hojas `chat`) |

In [ ]:
tel_a = setup_telemetry("supportops-agent", mode="memory", set_global=False)
tel_e = setup_telemetry("supportops-evaluator", mode="memory", set_global=False)
ev = OnlineEvaluator(JUDGE, StratifiedSampler(), tel_e)
col2 = CaseCollector(on_case=ev.submit)                         # al terminar cada ticket, el evaluador decide si lo muestrea
agent2 = SupportAgent(instrumentor=CompositeInstrumentor(OTelInstrumentor(tel_a), col2))      # el de trazas PRIMERO: así el span está activo cuando col2 lee los ids
res2 = run_tickets(agent2, generate_tickets(120, seed=SEED, fault_rates=FAULTS))
ev.run_pending(); tel_a.force_flush(); tel_e.force_flush()
print(f"{len(res2)} tickets · {len(ev.records)} evaluados · spans del agente {len(tel_a.span_records())} · spans del evaluador {len(tel_e.span_records())}")

In [ ]:
from supportops.regression import event_records
EVENTS = event_records(tel_e)
agent_spans = {s["span_id"]: s for s in tel_a.span_records()}
e = EVENTS[0]
print("evento:", json.dumps({k: v for k, v in e["attributes"].items()}, ensure_ascii=False, indent=1))
tgt = agent_spans[e["span_id"]]
print("\nel evento cuelga del span:", tgt["name"], "· trace_id coincide:", tgt["trace_id"] == e["trace_id"])

### Validación contra la spec

Escribimos el validador una vez y lo ejecutamos sobre **todos** los eventos: ¿existen los atributos en la instantánea?, ¿son del tipo correcto?, ¿están los *Required*?

In [ ]:
def validate_eval_event(attrs: dict) -> list[str]:
    """Problemas de un evento gen_ai.evaluation.result frente a la instantánea de la spec."""
    probs = []
    if "gen_ai.evaluation.name" not in attrs: probs.append("falta gen_ai.evaluation.name (Required)")
    if "gen_ai.evaluation.score.value" not in attrs and "gen_ai.evaluation.score.label" not in attrs: probs.append("falta score.value o score.label")
    for k, v in attrs.items():
        if k.startswith("gen_ai."):
            if k not in snap["attributes"]: probs.append(f"{k} no existe en la spec")
            elif snap["attributes"][k]["type"] == "double" and not isinstance(v, float): probs.append(f"{k} debe ser double")
            elif snap["attributes"][k]["type"] == "string" and not isinstance(v, str): probs.append(f"{k} debe ser string")
    return probs
bad = [(i, validate_eval_event(x["attributes"])) for i, x in enumerate(EVENTS) if validate_eval_event(x["attributes"])]
print(f"{len(EVENTS)} eventos validados · con problemas: {len(bad)}")
print("un evento sin name ->", validate_eval_event({"gen_ai.evaluation.score.value": 1.0}), "· con un atributo inventado ->", validate_eval_event({"gen_ai.evaluation.name": "x", "gen_ai.evaluation.score": 1.0, "gen_ai.evaluation.score.value": 1.0}))

In [ ]:
ev_spans = [s for s in tel_e.span_records() if s["name"] == "supportops.evaluate"]
one = ev_spans[0]
trace_spans = [s for s in tel_e.span_records() if s["trace_id"] == one["trace_id"]]
fig, ax = plt.subplots(figsize=(10, 2.4)); viz.waterfall(trace_spans, ax=ax, title="Traza de UNA evaluación (servicio supportops-evaluator)"); plt.tight_layout(); plt.show()
print("atributos del span evaluate:", {k: v for k, v in one["attributes"].items() if k.startswith("supportops.eval")})

> 👀 **Qué debes observar**
>
> - La cascada tiene dos spans: `supportops.evaluate` (el trabajo de evaluar) y su hijo `chat mock-large` (la llamada al juez, con tokens). Su duración es la **latencia del juez**, no la del usuario: la evaluación no está en el camino crítico.
> - Cada evento apunta (`trace_id`, `span_id`) al span `chat` del **agente** que produjo la respuesta evaluada: en Grafana, desde una puntuación mala abres la traza original con un clic.
> - En el span `supportops.evaluate` hay un *link* al span evaluado (no es su hijo: se evalúa minutos después). Un trace con la evaluación como hijo alargaría artificialmente la duración de la petición.

---
# 8 · Con el stack real: Tempo, Loki, Prometheus y Grafana

Con el stack (`docker compose --profile core --profile metrics up -d`) enviamos **60 s de tráfico en tiempo real** por `LivePipeline` (agente con métricas del módulo 09 + evaluador) y consultamos cada backend.
Sin stack esta celda se salta (modo degradado) y las explicaciones siguen valiendo.

In [ ]:
from supportops.regression import LivePipeline
from supportops.slo import BASELINE_FAULTS
LIVE_S = 45 if FAST_DEV_RUN else 120
if LIVE:
    lp = LivePipeline(("v1",), mode="otlp", sampler=StratifiedSampler(), metric_interval_s=5)
    live_tk = generate_tickets(int(LIVE_S * 3), seed=SEED, duration_s=LIVE_S, fault_rates={**BASELINE_FAULTS, "hallucination": 0.08}, start_s=0.0)
    t0 = time.time(); lp.run_live(live_tk, LIVE_S); lp.close()
    print(f"enviados {len(live_tk)} tickets en {time.time() - t0:.0f}s · evaluados {len(lp.ev.records)} · estimación de calidad {lp.ev.estimate_quality()['estimate']:.3f}")
    time.sleep(12)                       # métricas: scrape de 15 s; trazas: ≈ 15-20 s hasta ser buscables
else:
    print("Modo degradado: sin Collector no se envía nada. Levanta el stack con:", STACK.command)

In [ ]:
if LIVE:
    from supportops.otel_metrics import prom_instant
    qs = {"calidad groundedness (5 m)": 'sum(rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness",gen_ai_evaluation_score_label="pass"}[5m])) / sum(rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness"}[5m]))',
          "cobertura: evaluadas/pobl.": "sum(increase(supportops_eval_sampled_total[5m])) / sum(increase(supportops_eval_population_total[5m]))",
          "coste del juez (USD, 5 m)": "sum(increase(supportops_eval_cost_USD_total[5m]))"}
    for k, q in qs.items():
        r = prom_instant(q); print(f"{k:32s}", [round(x['value'], 4) for x in r if x['value'] is not None] if r else "sin datos aún")
    ta = backends.tempo_search('{ resource.service.name = "supportops-evaluator" && name = "supportops.evaluate" }', minutes=10, limit=5)
    print("Tempo · trazas de evaluación:", len(ta.get("traces", [])) if isinstance(ta, dict) else ta)
    lg = backends.loki_query('{service_name="supportops-evaluator"} | gen_ai_evaluation_name="groundedness" | gen_ai_evaluation_score_label="fail"', minutes=10, limit=3)
    print("Loki · evaluaciones fallidas de groundedness:", len(lg), "líneas · primera:", str(lg[0])[:260] if lg else "(aún ninguna)")

> **🧪 Experimento · Experimento medido (oct-2026, Loki 3.7.8)**
>
> El **nombre del evento** (`event_name`) **no llega** a Loki como etiqueta ni como metadato estructurado en la ruta OTLP: el cuerpo (`groundedness=fail`) y los atributos `gen_ai_evaluation_*`
> sí. Por eso las consultas filtran por `gen_ai_evaluation_name` y no por el nombre del evento, y por eso conviene poner un **cuerpo legible** en el evento. Los puntos de los atributos se convierten en `_`.
> El `trace_id`/`span_id` del evento son los del **span evaluado** (comprobado).

### El dashboard `14-calidad-online` (provisionado como código)

`stack/grafana/dashboards/14-calidad-online.json` (uid `calidad-online`) lo genera este mismo builder con `supportops.dashkit`. Paneles y consultas clave:

| Panel | Consulta |
|---|---|
| Calidad estimada (post-estratificada) | `sum(rate(population)·pass/evaluados por estrato)/sum(rate(population))` — la fórmula de la sección 6 en PromQL |
| Calidad por versión | `sum by (gen_ai_agent_version)(rate(results{pass}))/sum by (...)(rate(results))` |
| Cobertura de la evaluación | `increase(sampled)/increase(population)` por estrato |
| Distribución de puntuaciones | heatmap de `supportops_eval_score_bucket` |
| Feedback +/− | `supportops_feedback_total` por valoración |
| Coste del juez frente al del agente | `eval_cost / llm_cost` |
| Evaluaciones fallidas | LogQL `{service_name="supportops-evaluator"} \| gen_ai_evaluation_score_label="fail"` |
| Trazas evaluadas con fallo | TraceQL `{ name="supportops.evaluate" && span.supportops.eval.passed=false }` |

**Qué debes ver en Grafana:** un panel «Calidad estimada» estable alrededor del 0,85–0,95, que baja cuando inyectas alucinaciones; la tabla de logs lista las evaluaciones fallidas y un clic en `trace_id` abre la traza del agente.

---
# 9 · Feedback de usuario y coste de evaluar

## 9.1 El feedback explícito (+/−) es barato… y sesgado

En SupportOps el widget de chat permite valorar la respuesta. **No hay convención OTel** para feedback de usuario (a fecha de hoy): se modela como métrica de baja cardinalidad (`supportops.feedback{rating}`) y, si se guarda el texto, como log con el `trace_id` de la traza valorada.
Aquí lo **simulamos** (sintético): quien tuvo una mala experiencia responde ~5× más que quien no.

In [ ]:
FB = simulate_feedback(CASES, seed=SEED)
by = {f.ticket_id: f.rating for f in FB}
bad_t = [c for c in CASES if c.truth_bad]; good_t = [c for c in CASES if not c.truth_bad]
resp_bad = np.mean([c.ticket_id in by for c in bad_t]); resp_good = np.mean([c.ticket_id in by for c in good_t])
down_share = np.mean([f.rating < 0 for f in FB])
print(f"{len(FB)} valoraciones de {len(CASES)} tickets ({len(FB) / len(CASES):.1%}) · responden el {resp_bad:.0%} de los malos y el {resp_good:.1%} de los buenos")
print(f"− entre las valoraciones: {down_share:.0%}  ·  tickets realmente malos: {len(bad_t) / len(CASES):.0%}  ->  el feedback SOBRESTIMA el problema {down_share / (len(bad_t) / len(CASES)):.1f}×")
# ¿cuánta de la alucinación real captura el feedback frente al juez?
hall = [c for c in CASES if not c.truth["grounded"]]
print(f"alucinaciones con −: {np.mean([by.get(c.ticket_id) == -1 for c in hall]):.0%} · detectadas por el juez imperfecto: {np.mean([JUDGE.evaluate(c).scores['groundedness'].label == 'fail' for c in hall]):.0%}")
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(["tickets malos\n(verdad)", "− entre\nvaloraciones", "alucinaciones\ncon −", "alucinaciones\ndetectadas (juez)"], [len(bad_t) / len(CASES), down_share, np.mean([by.get(c.ticket_id) == -1 for c in hall]), np.mean([JUDGE.evaluate(c).scores['groundedness'].label == 'fail' for c in hall])],
       color=[viz.PALETTE["ok"], viz.PALETTE["warn"], viz.PALETTE["warn"], viz.PALETTE["profiles"]]); ax.set_ylim(0, 1); ax.set_title("Feedback vs juez frente a la verdad")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - Solo ≈ 8 % de los tickets recibe valoración y el porcentaje de valoraciones negativas **sobrestima** la tasa real de problemas ≈ 3×: responden los enfadados (sesgo de autoselección).
> - Y lo importante: el feedback **no ve** casi ninguna alucinación (≈ 12 % reciben un negativo; el cliente que recibe un dato falso verosímil no protesta hasta que le falla). El juez imperfecto ve ≈ 90 %.
> - Uso correcto del feedback: **disparador** (un − fuerza la evaluación del ticket y entra en el dataset de regresión, módulo 15) y señal de tendencia, **no** estimador de la calidad poblacional.

## 9.2 El coste de evaluar: ¿cuánto muestrear?

Cada evaluación es una llamada a un modelo (el prompt incluye rúbrica + contexto + respuesta). Con los precios **ilustrativos** del mock (juez `mock-large`, agente `mock-small`):

In [ ]:
ev_all = run_sampler(RateSampler(1.0), judge=JUDGE)
judge_per = np.mean([r.verdict.cost_usd for r in ev_all.records]); agent_per = np.mean([r.case.cost_usd for r in ev_all.records])
print(f"coste medio por ticket: agente ${agent_per:.6f} · juez ${judge_per:.6f} -> evaluar UN ticket cuesta {judge_per / agent_per:.1f}× lo que cuesta atenderlo")
rates = np.array([0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0]); p = 0.85; N_DAY = 20000
n_eval = rates * N_DAY
cost_day = n_eval * judge_per; width = 1.96 * np.sqrt(p * (1 - p) / n_eval)
fig, ax = plt.subplots(figsize=(8, 4)); ax2 = ax.twinx()
ax.plot(rates * 100, cost_day / (N_DAY * agent_per) * 100, "o-", color=viz.PALETTE["metrics"], label="coste del juez / coste del agente (%)"); ax.set_xlabel("% de tickets evaluados"); ax.set_ylabel("sobrecoste sobre el agente (%)", color=viz.PALETTE["metrics"])
ax2.plot(rates * 100, width * 100, "s--", color=viz.PALETTE["traces"], label="± puntos de error (IC95) de la calidad"); ax2.set_ylabel("± puntos de error del IC95", color=viz.PALETTE["traces"]); ax2.grid(False)
ax.set_xscale("log"); ax.set_title("Coste vs precisión según la tasa de muestreo (20 000 tickets/día)")
plt.tight_layout(); plt.show()
print({f"{r:.0%}": f"{c / (N_DAY * agent_per):.0%} de sobrecoste, ±{w * 100:.1f} pt" for r, c, w in zip(rates, cost_day, width)})

> 👀 **Qué debes observar**
>
> - Evaluar un ticket con un juez grande cuesta **~9× más** que atenderlo con el modelo pequeño. Evaluarlo todo multiplicaría la factura de IA por ≈ 10: inviable.
> - La precisión mejora con la **raíz** de n: del 1 % al 10 % de muestreo el error baja de ±~5 a ±~1,6 puntos; pasar del 10 % al 100 % solo lo baja de ±1,6 a ±0,5 por 10× el dinero. **El codo de la curva está en torno al 5–10 %.**
> - Palancas para abaratar sin perder señal: (1) **cascada** (reglas primero, LLM solo en lo dudoso), (2) juez **más pequeño** calibrado contra el grande, (3) un solo prompt con **todos los criterios**, (4) muestrear por estratos de valor, (5) *caching* de la rúbrica (módulo 13).

---
# 10 · 🏭 En producción

- **Evaluación como consumidor de trazas.** Las plataformas de observabilidad de LLMs (Langfuse, LangSmith, Arize Phoenix y otras) comparten el patrón: puntuaciones ligadas a una traza/span, evaluadores asíncronos sobre una muestra y datasets creados desde producción. El valor del enfoque OTel es que las puntuaciones viven en **los mismos backends** que el resto de señales y se correlacionan con ellas.
- **Dónde se ejecuta el juez.** Servicio independiente con su propia cuota, cola y escalado; si el proveedor del juez cae, la *evaluación* se retrasa, el usuario no se entera. Backpressure: descarta evaluaciones antiguas antes que bloquear.
- **Versiona todo lo que influye en la nota**: versión del prompt del juez, del modelo juez y de la rúbrica (`supportops.eval.judge`). Cambiar el juez rompe la comparabilidad histórica de la serie.
- **Calibración continua.** Muestra semanalmente casos para revisión humana, recalcula sensibilidad/especificidad y detecta *drift del juez* (un proveedor cambia el modelo por debajo).
- **Cardinalidad.** `criterio × etiqueta × estrato × versión` son unas decenas de series: seguro. `ticket_id`, `customer_id`, el texto de la explicación **nunca** como etiqueta (van en el log/span).

---
# 11 · 🧠 Secretos de la élite

1. **La métrica de la muestra no es la métrica de la población.** Cualquier muestreo no uniforme (errores, premium, −) exige guardar la probabilidad de inclusión y **post-estratificar** (sección 6); alertar sobre la media ingenua crea una «calidad baja» permanente que es un artefacto. Mide `population` para *todas* las peticiones, no solo las evaluadas.
2. **Corrige la tasa con la sensibilidad y la especificidad del juez** (Rogan & Gladen, 1978): con prevalencia baja, un 2 % de falsos positivos puede ser la mayoría de las alarmas. Una nota sin el error del juez no es una medida, es una opinión.
3. **Sondas de sesgo por cada cambio de juez.** Relleno (verbosidad), permutación de orden (posición) y, si juez y agente son de la misma familia, autopreferencia (Panickssery et al., 2024). Es un test de regresión *del evaluador*, con la misma disciplina que el del agente.
4. **Código antes que LLM para todo lo verificable.** PII, ids, importes y formatos se comprueban con regex/esquemas: coste ≈ 0, reproducibles y sin enviar datos personales a un tercero. El juez LLM se reserva para lo semántico.
5. **El evaluador es otro servicio y su traza va enlazada, no anidada.** Así la duración de la petición no incluye la evaluación, el coste del juez no contamina el coste del agente (la regla de «solo hojas `chat`» se aplica *por servicio*) y se puede escalar/limitar por separado.
6. **Muestreo determinista por hash del id**, no `random()`: la decisión es reproducible, idéntica en todas las réplicas y estable ante reintentos; puedes re-evaluar *exactamente* la misma muestra con un juez nuevo.
7. **Cuenta los fallos del juez.** `parse_error`, timeouts y *rate limits* del juez deben ser una métrica y una alerta: si no, la calidad «mejora» el día que el juez se rompe porque solo puntúas lo que sí parsea.
8. **No alertes sobre una nota individual ni sobre una media sin intervalo.** Alerta sobre una **tasa** en una ventana con suficiente n (módulo 16: `min_events`) y comprueba que el IC no solape el umbral.
9. **El feedback de usuario es un disparador, no un estimador**: sesgo de autoselección (responden los enfadados) y ceguera ante errores verosímiles.

---
# 12 · ⚠️ Errores comunes

- Evaluar con el **mismo modelo** (o familia) que genera la respuesta y no medir la autopreferencia.
- Poner la nota del juez como etiqueta de una métrica (alta cardinalidad) en lugar de histograma + etiqueta pass/fail.
- Usar el **puntaje medio** de una escala de 1–10: poco fiable e inestable; prefiere pass/fail por criterio con rúbrica concreta, o escalas pequeñas con ejemplos.
- Mandar la conversación completa (con PII) al juez de un tercero sin redactar.
- Cambiar el prompt del juez sin versionarlo ni recalibrar: la serie histórica pierde sentido.
- Evaluar *dentro* del camino de la petición (latencia para el usuario) cuando no hace falta bloquear la respuesta.
- Confundir **correlación con la satisfacción** con **corrección**: una respuesta amable y falsa puede puntuar bien.

---
# 13 · Autoevaluación

<details>
<summary>1 · (Cálculo) Con una sensibilidad de 0,90 y una especificidad de 0,98, el juez marca el 11 % de las respuestas como alucinadas. ¿Cuál es la tasa real de alucinación?</summary>

Rogan–Gladen: \( (0{,}11 + 0{,}98 - 1)/(0{,}90 + 0{,}98 - 1) = 0{,}09/0{,}88 \approx 0{,}102 \). La tasa real es ≈ **10,2 %**, no 11 %: parte de las alarmas son falsos positivos y se escapan alucinaciones, y el efecto neto depende de la prevalencia.

</details>

<details>
<summary>2 · (Cálculo) Población: base 8 000 (evaluados 400, 360 buenos), error 200 (evaluados 200, 100 buenos). Estima la tasa de calidad OK poblacional y compárala con la media de la muestra.</summary>

Post-estratificado: \( (8000/8200)\cdot 0{,}9 + (200/8200)\cdot 0{,}5 = 0{,}878 + 0{,}0122 \approx 0{,}890 \). La media ingenua de la muestra es \( (360+100)/600 = 0{,}767 \): subestima 12 puntos porque el estrato «error» está sobrerrepresentado (200 de 600 evaluados, pero solo 2,4 % de la población).

</details>

<details>
<summary>3 · (Diagnóstico) Tras cambiar el modelo juez, la «calidad» de SupportOps sube 4 puntos de un día para otro sin ningún despliegue del agente. ¿Qué sospechas y cómo lo compruebas?</summary>

Cambió la **vara de medir**, no el agente: otro juez con distinta sensibilidad/especificidad o sesgo (p. ej. más indulgente con respuestas largas). Comprueba (a) `supportops.eval.judge`/versión en las métricas, (b) re-evalúa una muestra congelada con ambos jueces (comparación pareada), (c) recalcula sensibilidad/especificidad con la muestra etiquetada, (d) mira `parse_error`. Nunca compares series de jueces distintos sin calibrar.

</details>

<details>
<summary>4 · (Diagnóstico) El panel de calidad muestra 100 % durante una hora; luego se recupera «de golpe» a 94 %. ¿Qué mirarías antes de celebrar?</summary>

Que el juez no estuviera caído: `supportops.eval.errors`/`parse_error`, la métrica `sampled` frente a `results` (¿se evaluó algo?), la cobertura y el volumen `n` por ventana. Un 100 % con n≈0 o con el juez fallando es un **falso verde**. Por eso el panel de calidad lleva siempre al lado la cobertura y el nº de evaluaciones.

</details>

<details>
<summary>5 · (Transferencia) Tu equipo de soporte humano ya etiqueta 50 tickets a la semana. ¿Cómo los usas para mantener sano el sistema de evaluación?</summary>

Como **conjunto de calibración**: cada semana, el juez evalúa esos 50 y se recalculan sensibilidad/especificidad con su IC (Wilson); si caen por debajo de un umbral se bloquea el uso del juez para decisiones. También como semilla del dataset de oro del módulo 15, y para medir el desacuerdo entre anotadores (el techo de fiabilidad del juez).

</details>

<details>
<summary>6 · (Transferencia) ¿Cómo adaptarías este diseño a un agente que genera código o consultas SQL?</summary>

Priorizando evaluadores **ejecutables**: compilar/ejecutar en un sandbox, comprobar el plan, comparar resultados con una consulta de referencia. El juez LLM queda para estilo y seguridad. El mismo esqueleto: evento de evaluación ligado al span, muestreo por estratos (consultas lentas, que tocan tablas sensibles) y métricas de pass-rate.

</details>

<details>
<summary>7 · ¿Por qué el evento de evaluación lleva el `trace_id`/`span_id` del span evaluado y no los del span `supportops.evaluate`?</summary>

Porque la spec pide que el evento cuelgue del span de la operación **evaluada**: así una consulta por `trace_id` en Loki devuelve, junto a los logs del agente, sus puntuaciones, y desde la traza del agente se llega a ellas. El span del evaluador tiene su propia traza y un *link* al evaluado.

</details>

---
# 14 · Referencias

- OpenTelemetry, *Semantic conventions for generative AI* (repo `open-telemetry/semantic-conventions-genai`, estado *Development*): evento `gen_ai.evaluation.result` — https://github.com/open-telemetry/semantic-conventions-genai (`docs/gen-ai/gen-ai-events.md`).
- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* arXiv:2306.05685 — sesgos de posición, verbosidad y autopreferencia.
- Panickssery, A., Bowman, S. R., Feng, S. (2024). *LLM Evaluators Recognize and Favor Their Own Generations.* arXiv:2404.13076 (NeurIPS 2024).
- Rogan, W. J., Gladen, B. (1978). *Estimating prevalence from the results of a screening test.* American Journal of Epidemiology 107(1):71–76.
- Husain, H. (2024). *Your AI Product Needs Evals.* https://hamel.dev/blog/posts/evals/
- Yan, E. (2024). *Evaluating the Effectiveness of LLM-Evaluators (aka LLM-as-Judge).* https://eugeneyan.com/writing/llm-evaluators/
- Es, S. et al. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation.* arXiv:2309.15217 — fidelidad (*faithfulness*) y relevancia sin referencia.
- Cochran, W. G. (1977). *Sampling Techniques* (3.ª ed.), Wiley — muestreo estratificado y estimadores post-estratificados.
- Grafana Labs, documentación de *Alerting* y *Dashboards* (provisioning como código) — https://grafana.com/docs/grafana/latest/